In [ ]:

import glob
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord
import astropy.units as u
from regions import Regions
import datetime
import os
filtername = 'f480m'
image_filenames ={
    "f140m": "/orange/adamginsburg/jwst/w51/F140M/pipeline/jw06151-o001_t001_nircam_clear-f140m-merged_i2d.fits",
    "f150w": "/orange/adamginsburg/jwst/w51/F150W/pipeline/jw06151-o001_t001_nircam_clear-f150w-merged_i2d.fits",
    "f162m": "/orange/adamginsburg/jwst/w51/F162M/pipeline/jw06151-o001_t001_nircam_clear-f162m-merged_i2d.fits",
    "f182m": "/orange/adamginsburg/jwst/w51/F182M/pipeline/jw06151-o001_t001_nircam_clear-f182m-merged_i2d.fits",
    "f187n": "/orange/adamginsburg/jwst/w51/F187N/pipeline/jw06151-o001_t001_nircam_clear-f187n-merged_i2d.fits",
    "f210m": "/orange/adamginsburg/jwst/w51/F210M/pipeline/jw06151-o001_t001_nircam_clear-f210m-merged_i2d.fits",
    "f335m": "/orange/adamginsburg/jwst/w51/F335M/pipeline/jw06151-o001_t001_nircam_clear-f335m-merged_i2d.fits",
    "f360m": "/orange/adamginsburg/jwst/w51/F360M/pipeline/jw06151-o001_t001_nircam_clear-f360m-merged_i2d.fits",
    "f405n": "/orange/adamginsburg/jwst/w51/F405N/pipeline/jw06151-o001_t001_nircam_clear-f405n-merged_i2d.fits",
    "f410m": "/orange/adamginsburg/jwst/w51/F410M/pipeline/jw06151-o001_t001_nircam_clear-f410m-merged_i2d.fits", # weird, the filename is different from what is downloaded with the STScI pipeline...
    "f480m": "/orange/adamginsburg/jwst/w51/F480M/pipeline/jw06151-o001_t001_nircam_clear-f480m-merged_i2d.fits",
    "f560w": "/orange/adamginsburg/jwst/w51/F560W/pipeline/jw06151-o002_t001_miri_f560w_i2d.fits",
    "f770w": "/orange/adamginsburg/jwst/w51/F770W/pipel ine/jw06151-o002_t001_miri_f770w_i2d.fits",
    "f1000w": "/orange/adamginsburg/jwst/w51/F1000W/pipeline/jw06151-o002_t001_miri_f1000w_i2d.fits",
    "f1280w": "/orange/adamginsburg/jwst/w51/F1280W/pipeline/jw06151-o002_t001_miri_f1280w_i2d.fits",
    "f1500w": "/orange/adamginsburg/jwst/w51/F1500W/pipeline/jw06151-o002_t001_miri_f1500w_i2d.fits",
    "f2100w": "/orange/adamginsburg/jwst/w51/F2100W/pipeline/jw06151-o002_t001_miri_f2100w_i2d.fits",
    
}

tblnames = glob.glob(f'/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/{filtername.upper()}/*nrca*_combined_with_satstars.fits')
print(f'Found {len(tblnames)} tables to combine.')
for tbl in tblnames:
    print(tbl)  
    #print modified time of the file
    print(f"Modified time: {datetime.datetime.fromtimestamp(os.path.getmtime(tbl))}")
for ii, tblname in enumerate(tblnames):
    if ii==0:
        base_tbl = Table.read(tblname)
    else:
        new_tbl = Table.read(tblname)
        base_tbl = vstack([base_tbl, new_tbl])
skycoord = base_tbl['skycoord_centroid']

merged_tblnames = glob.glob(f"/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/{filtername.lower()}_nrca_indivexp_merged_dao_after_merger_combined_with_satstars.fits")
merged_tbl = Table.read(merged_tblnames[0])
print('merged_tblnames', merged_tblnames)
print(merged_tbl.colnames)
print(datetime.datetime.fromtimestamp(os.path.getmtime(merged_tblnames[0])))
merged_skycoord = merged_tbl['skycoord']
nmatch = merged_tbl['nmatch']



In [ ]:
from astropy.io import fits
from astropy.wcs import WCS
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from regions import Regions
from astropy.coordinates import SkyCoord
import numpy as np
from astropy.nddata import Cutout2D
def load_data(filename):
    fh = fits.open(filename)
    im1 = fh
    data = im1['SCI'].data
    try:
        wht = im1['WHT'].data
    except KeyError:
        wht = None
    err = im1['ERR'].data
    instrument = im1[0].header['INSTRUME']
    telescope = im1[0].header['TELESCOP']
    obsdate = im1[0].header['DATE-OBS']
    return fh, im1, data, wht, err, instrument, telescope, obsdate
img_filename = image_filenames[filtername]
fh, im1, img_data, wht, err, instrument, telescope, obsdate = load_data(img_filename)
wcs = WCS(im1[1].header)

pixcoord = wcs.world_to_pixel(skycoord)
merged_pixcoord = wcs.world_to_pixel(merged_skycoord)



test_regions = Regions.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/f140m_refine_test.reg', format='crtf')
by_eye_sources = Regions.read(f'/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/by_eye_star_f480m.reg', format='crtf')
ra_reg = []
dec_reg = []
for reg_ in by_eye_sources:
    ra_reg.append(reg_.center.ra.deg)
    dec_reg.append(reg_.center.dec.deg)
by_eye_skycoord = SkyCoord(ra=ra_reg*u.deg, dec=dec_reg*u.deg, frame='icrs')

i2d, d2d, _ = by_eye_skycoord.match_to_catalog_sky(merged_skycoord, nthneighbor=1)
matched = d2d < 0.1*u.arcsec

nmatch_for_matched = nmatch[i2d[matched]]
nmatch_2_idx = np.where(nmatch_for_matched == 3)[0]
by_eye_pixcoord = wcs.world_to_pixel(by_eye_skycoord)

norm = simple_norm(img_data, stretch='sqrt', percent=99)


for ii, idx in enumerate(i2d[matched][nmatch_2_idx]):
    if ii<=2:

        fig = plt.figure(figsize=(10,10))
        ax = fig.add_subplot(111,)
        ax.imshow(img_data, origin='lower', cmap='gray', norm=norm)
        # get pixcoord, merged_pixcoord, by_eye pixcoord that are within 30 pixels of the merged_pixcoord[idx]
        idx_within = np.where((np.abs(pixcoord[0]-merged_pixcoord[0][idx])<30) & (np.abs(pixcoord[1]-merged_pixcoord[1][idx])<30))[0]
        ax.scatter(pixcoord[0][idx_within], pixcoord[1][idx_within], s=50, edgecolor='red', facecolor='none', label='Before merging')
        idx_within = np.where((np.abs(merged_pixcoord[0]-merged_pixcoord[0][idx])<30) & (np.abs(merged_pixcoord[1]-merged_pixcoord[1][idx])<30))[0]
        ax.scatter(merged_pixcoord[0][idx_within], merged_pixcoord[1][idx_within], s=50, edgecolor='blue', facecolor='none', label='After merging')
        for jj in idx_within:
            ax.annotate(f'{nmatch[jj]}', (merged_pixcoord[0][jj], merged_pixcoord[1][jj]), color='cyan', fontsize=8)
        idx_within = np.where((np.abs(by_eye_pixcoord[0]-merged_pixcoord[0][idx])<30) & (np.abs(by_eye_pixcoord[1]-merged_pixcoord[1][idx])<30))[0]
        ax.scatter(by_eye_pixcoord[0][idx_within], by_eye_pixcoord[1][idx_within], s=200, edgecolor='cyan', facecolor='none', label='By eye sources')
        
        ax.legend(loc='upper right')
        ax.set_title(f'{filtername.upper()} - Before and After Merging')
        ax.set_xlim(merged_pixcoord[0][idx]-5, merged_pixcoord[0][idx]+5)
        ax.set_ylim(merged_pixcoord[1][idx]-5, merged_pixcoord[1][idx]+5)
        plt.show()
